In [13]:
# ================================================================
# 0. Colab setup and imports
# ================================================================
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime
import warnings
warnings.filterwarnings('ignore', category=FutureWarning)

PROJECT_ROOT = Path('/content/drive/MyDrive/Vol_fitter')
RAW_DIR = PROJECT_ROOT / 'data_SPY/'                    # OptionsDX yearly files, immutable
PROCESSED_DIR = PROJECT_ROOT / 'data_processed'         # per-date post de-Americanisation
FITTED_DIR = PROJECT_ROOT / 'data_fitted'               # per-date post SVI/SSVI fit
AUX_DIR = PROJECT_ROOT / 'data_raw_aux'                 # Treasury curve, event calendar

for d in [PROCESSED_DIR, FITTED_DIR, AUX_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f'Raw dir:       {RAW_DIR}    ({len(list(RAW_DIR.glob("*.parquet")))} yearly files)')
print(f'Processed dir: {PROCESSED_DIR}')
print(f'Fitted dir:    {FITTED_DIR}')
print(f'Aux dir:       {AUX_DIR}')

Created volsurf package:
  volsurf/__init__.py  (34 bytes)
  volsurf/data.py  (715 bytes)
  volsurf/paths.py  (376 bytes)
  volsurf/rates.py  (1237 bytes)


In [3]:
# ================================================================
# 1. Schema consistency check across all 14 years
# ================================================================
# Before writing any pipeline, verify the OptionsDX schema is stable across years.
# Schema drift between 2010 and 2023 would silently break a one-year-tested loader.

schemas = {}
for year in range(2010, 2024):
    fp = RAW_DIR / f'spy_eod_{year}.parquet'
    if not fp.exists():
        print(f'{year}: MISSING')
        continue
    # Read just the schema, no rows
    schema = pd.read_parquet(fp, columns=None).head(0)
    cols = tuple(c.strip('[]') for c in schema.columns)
    schemas[year] = cols

# Are all schemas identical?
unique_schemas = set(schemas.values())
print(f'Years loaded: {len(schemas)}')
print(f'Unique schemas: {len(unique_schemas)}')

if len(unique_schemas) == 1:
    print('All years share an identical schema. Loader will work uniformly.')
    print(f'Columns ({len(list(unique_schemas)[0])}):')
    for c in list(unique_schemas)[0]:
        print(f'  {c}')
else:
    # Surface the differences so you can patch the loader
    base = schemas[2010]
    for year, cols in schemas.items():
        added = set(cols) - set(base)
        removed = set(base) - set(cols)
        if added or removed:
            print(f'{year}: added={sorted(added)} removed={sorted(removed)}')

Years loaded: 14
Unique schemas: 1
All years share an identical schema. Loader will work uniformly.
Columns (33):
  QUOTE_UNIXTIME
  QUOTE_READTIME
  QUOTE_DATE
  QUOTE_TIME_HOURS
  UNDERLYING_LAST
  EXPIRE_DATE
  EXPIRE_UNIX
  DTE
  C_DELTA
  C_GAMMA
  C_VEGA
  C_THETA
  C_RHO
  C_IV
  C_VOLUME
  C_LAST
  C_SIZE
  C_BID
  C_ASK
  STRIKE
  P_BID
  P_ASK
  P_SIZE
  P_LAST
  P_DELTA
  P_GAMMA
  P_VEGA
  P_THETA
  P_RHO
  P_IV
  P_VOLUME
  STRIKE_DISTANCE
  STRIKE_DISTANCE_PCT


In [4]:
# ================================================================
# 2. Row counts and date coverage per year
# ================================================================
# Confirms each year has reasonable trading day coverage (250 ish) and we have
# the expected total corpus before processing.

stats = []
for year in range(2010, 2024):
    fp = RAW_DIR / f'spy_eod_{year}.parquet'
    if not fp.exists(): continue
    df = pd.read_parquet(fp, columns=['[QUOTE_DATE]', '[STRIKE]', '[UNDERLYING_LAST]'])
    df.columns = [c.strip('[]') for c in df.columns]
    n_rows = len(df)
    n_days = df.QUOTE_DATE.nunique()
    spot_min = df.UNDERLYING_LAST.min()
    spot_max = df.UNDERLYING_LAST.max()
    stats.append({
        'year': year, 'rows': n_rows, 'trading_days': n_days,
        'avg_rows_per_day': n_rows / n_days,
        'spot_min': spot_min, 'spot_max': spot_max,
    })

stats_df = pd.DataFrame(stats)
print(stats_df.to_string(index=False))
print(f'\nTotal rows across 2010-2023: {stats_df.rows.sum():,}')
print(f'Total trading days:          {stats_df.trading_days.sum():,}')

 year    rows  trading_days  avg_rows_per_day  spot_min  spot_max
 2010  262978           250       1051.912000    102.25    125.93
 2011  285265           245       1164.346939    109.91    136.42
 2012  340926           250       1363.704000    127.54    147.13
 2013  361258           252       1433.563492    145.53    184.46
 2014  421454           250       1685.816000    174.00    208.65
 2015  492643           252       1954.932540    187.28    213.45
 2016  544085           251       2167.669323    182.91    227.68
 2017  609530           250       2438.120000    225.17    268.17
 2018  755129           252       2996.543651    234.34    293.60
 2019  834075           248       3363.205645    244.09    322.94
 2020 1164401           250       4657.604000    222.21    373.93
 2021 1277698           252       5070.230159    368.84    477.48
 2022 1146980           256       4480.390625    356.58    477.77
 2023  972162           250       3888.648000    379.38    476.73

Total row

In [5]:
# ================================================================
# 3. The canonical loader function
# ================================================================
# Single source of truth for reading any OptionsDX yearly file. Strips bracket
# wrappers, parses dates, and adds derived columns. All downstream code uses
# this function rather than reading parquet directly.

def load_optionsdx_year(year: int, raw_dir: Path = RAW_DIR) -> pd.DataFrame:
    """Load one year of OptionsDX SPY EOD data with normalised schema.

    Returns a DataFrame with columns matching the OptionsDX schema, bracket
    wrappers stripped, dates parsed, and T_years derived from DTE.
    """
    fp = raw_dir / f'spy_eod_{year}.parquet'
    if not fp.exists():
        raise FileNotFoundError(f'No raw file for year {year}: {fp}')
    df = pd.read_parquet(fp)
    df.columns = [c.strip('[]') for c in df.columns]
    df['QUOTE_DATE'] = pd.to_datetime(df['QUOTE_DATE'])
    df['EXPIRE_DATE'] = pd.to_datetime(df['EXPIRE_DATE'])
    df['T_years'] = df['DTE'] / 365.0
    return df

# Smoke test on 2010
df_2010 = load_optionsdx_year(2010)
print(f'Loaded 2010: {len(df_2010):,} rows, '
      f'{df_2010.QUOTE_DATE.nunique()} trading days, '
      f'{df_2010.EXPIRE_DATE.nunique()} unique expiries')
print(f'\nFirst row:')
print(df_2010.iloc[0])

Loaded 2010: 262,978 rows, 250 trading days, 54 unique expiries

First row:
QUOTE_UNIXTIME                  1262638800
QUOTE_READTIME            2010-01-04 16:00
QUOTE_DATE             2010-01-04 00:00:00
QUOTE_TIME_HOURS                      16.0
UNDERLYING_LAST                     113.29
EXPIRE_DATE            2010-01-15 00:00:00
EXPIRE_UNIX                     1263589200
DTE                                   11.0
C_DELTA                            0.88304
C_GAMMA                            0.00005
C_VEGA                             0.02341
C_THETA                           -0.01037
C_RHO                               0.0025
C_IV                                2.9353
C_VOLUME                               0.0
C_LAST                               54.67
C_SIZE                           305 x 270
C_BID                                 58.2
C_ASK                                 58.4
STRIKE                                55.0
P_BID                                  0.0
P_ASK                

In [6]:
# ================================================================
# 4. Data quality baseline (2010 reference)
# ================================================================
# Quantify the junk in raw IVs and bid-ask quality. Replicating the inspection
# we did during planning so it lives in the notebook as a tracked diagnostic.

df = df_2010.copy()
n = len(df)

print('Raw IV quality:')
print(f'  C_IV NaN:    {df.C_IV.isna().sum():>7,} ({df.C_IV.isna().mean():.1%})')
print(f'  P_IV NaN:    {df.P_IV.isna().sum():>7,} ({df.P_IV.isna().mean():.1%})')
print(f'  C_IV > 5.0:  {(df.C_IV > 5.0).sum():>7,} ({(df.C_IV > 5.0).mean():.1%})')
print(f'  P_IV > 5.0:  {(df.P_IV > 5.0).sum():>7,} ({(df.P_IV > 5.0).mean():.1%})')
print(f'  C_IV < 0:    {(df.C_IV < 0).sum():>7,}  (inversion artefacts)')
print(f'  P_IV < 0:    {(df.P_IV < 0).sum():>7,}')

print('\nBid-ask quality:')
both_bid = (df.C_BID > 0) & (df.P_BID > 0)
print(f'  Both C_BID and P_BID > 0: {both_bid.sum():,} / {n:,} ({both_bid.mean():.1%})')

filt = df[both_bid & (df.C_ASK > df.C_BID) & (df.P_ASK > df.P_BID)]
per_slice = filt.groupby(['QUOTE_DATE', 'EXPIRE_DATE']).size()
print(f'\nAfter basic bid/ask filter:')
print(f'  Median strikes per (date, expiry): {per_slice.median():.0f}')
print(f'  Median expiries per date:          {filt.groupby("QUOTE_DATE").EXPIRE_DATE.nunique().median():.0f}')

Raw IV quality:
  C_IV NaN:     24,517 (9.3%)
  P_IV NaN:      4,244 (1.6%)
  C_IV > 5.0:      581 (0.2%)
  P_IV > 5.0:       86 (0.0%)
  C_IV < 0:      1,650  (inversion artefacts)
  P_IV < 0:      6,123

Bid-ask quality:
  Both C_BID and P_BID > 0: 201,557 / 262,978 (76.6%)

After basic bid/ask filter:
  Median strikes per (date, expiry): 56
  Median expiries per date:          15


# Trying to speed up the file read:

I am led to believe that I can change to local and thus read data much more quickly.

In [7]:
# ================================================================
# 5. Optional: stage raw files locally for fast iteration
# ================================================================
# Drive reads cost ~30-50ms per chunk; local /content reads cost <1ms.
# One-time copy. Re-runs are free since the destination check skips existing.

import shutil

LOCAL_RAW = Path('/content/data_SPY')
LOCAL_RAW.mkdir(exist_ok=True)

for src in sorted(RAW_DIR.glob('spy_eod_*.parquet')):
    dst = LOCAL_RAW / src.name
    if not dst.exists() or dst.stat().st_size != src.stat().st_size:
        print(f'Copying {src.name} ({src.stat().st_size / 1e6:.1f} MB)...')
        shutil.copy(src, dst)

print(f'\nStaged {len(list(LOCAL_RAW.glob("*.parquet")))} files to {LOCAL_RAW}')
print(f'Total: {sum(f.stat().st_size for f in LOCAL_RAW.glob("*.parquet")) / 1e6:.0f} MB')

# Switch the loader to local for the rest of the session
RAW_DIR_FAST = LOCAL_RAW

Copying spy_eod_2010.parquet (16.0 MB)...
Copying spy_eod_2011.parquet (17.8 MB)...
Copying spy_eod_2012.parquet (20.3 MB)...
Copying spy_eod_2013.parquet (21.0 MB)...
Copying spy_eod_2014.parquet (25.3 MB)...
Copying spy_eod_2015.parquet (29.5 MB)...
Copying spy_eod_2016.parquet (32.1 MB)...
Copying spy_eod_2017.parquet (36.0 MB)...
Copying spy_eod_2018.parquet (46.0 MB)...
Copying spy_eod_2019.parquet (53.8 MB)...
Copying spy_eod_2020.parquet (76.8 MB)...
Copying spy_eod_2021.parquet (85.6 MB)...
Copying spy_eod_2022.parquet (75.4 MB)...
Copying spy_eod_2023.parquet (60.6 MB)...

Staged 14 files to /content/data_SPY
Total: 596 MB


# Treasury Yield curve via FRED

In [10]:
# ================================================================
# 1. Treasury yield curve via FRED
# ================================================================
# daily history from 1962. Pull each constant-maturity
# series once for the full 2010-present window, persist as one parquet,
# reference by date during de-Americanisation.

!pip install -q fredapi

from fredapi import Fred
from google.colab import userdata

FRED_KEY = userdata.get('FRED_API_KEY')
fred = Fred(api_key=FRED_KEY)

# FRED series IDs for the daily constant-maturity Treasury yield curve.
# Maturities given in years for downstream interpolation.
TREASURY_SERIES = {
    'DGS1MO': 1/12,
    'DGS3MO': 0.25,
    'DGS6MO': 0.5,
    'DGS1':   1.0,
    'DGS2':   2.0,
    'DGS3':   3.0,
    'DGS5':   5.0,
    'DGS7':   7.0,
    'DGS10': 10.0,
    'DGS20': 20.0,
    'DGS30': 30.0,
}

frames = []
for series_id, maturity in TREASURY_SERIES.items():
    s = fred.get_series(series_id, observation_start='2010-01-01')
    df = s.rename('yield_pct').reset_index().rename(columns={'index': 'date'})
    df['series'] = series_id
    df['maturity_years'] = maturity
    frames.append(df)

treasury = pd.concat(frames, ignore_index=True)
treasury['date'] = pd.to_datetime(treasury['date'])

# FRED uses NaN for holidays and missing observations. Forward-fill within
# each series to bridge bond-market holidays that do not align with equity holidays.
treasury = treasury.sort_values(['series', 'date'])
treasury['yield_pct'] = treasury.groupby('series')['yield_pct'].ffill(limit=5)
treasury = treasury.dropna(subset=['yield_pct'])

print(f'Treasury data: {len(treasury):,} rows')
print(f'Date range: {treasury.date.min()} to {treasury.date.max()}')
print(f'Series counts:')
print(treasury.groupby('series').size())

Treasury data: 47,124 rows
Date range: 2010-01-04 00:00:00 to 2026-06-04 00:00:00
Series counts:
series
DGS1      4284
DGS10     4284
DGS1MO    4284
DGS2      4284
DGS20     4284
DGS3      4284
DGS30     4284
DGS3MO    4284
DGS5      4284
DGS6MO    4284
DGS7      4284
dtype: int64


In [11]:
# ================================================================
# 2.1 Build the interpolated curve lookup function
# ================================================================
# Given a date and maturity in years, return the continuously-compounded
# risk-free rate. Linear interpolation in maturity, flat extrapolation
# below 1mo and above 30y.

from scipy.interpolate import interp1d

def get_rate(date: pd.Timestamp, T_years: float, treasury_df: pd.DataFrame) -> float:
    """Continuously-compounded risk-free rate for given date and maturity.

    Treasury yields are quoted bond-equivalent (BEY). For BS2002 and Black-Scholes
    we want continuous compounding: r_cc = log(1 + r_bey/2) * 2 approximately.
    For yields below ~10%, r_cc approx r_bey to within 5 bps, so we use the
    BEY directly. Refine if needed.
    """
    day = treasury_df[treasury_df.date == pd.Timestamp(date)]
    if day.empty:
        # Try forward fill within 5 days
        day = treasury_df[(treasury_df.date <= pd.Timestamp(date)) &
                          (treasury_df.date >= pd.Timestamp(date) - pd.Timedelta(days=5))]
        if day.empty:
            return np.nan
        day = day[day.date == day.date.max()]
    day = day.sort_values('maturity_years')
    if len(day) < 2: return np.nan
    f = interp1d(day.maturity_years.values, day.yield_pct.values / 100.0,
                 kind='linear', bounds_error=False,
                 fill_value=(day.yield_pct.values[0] / 100.0,
                             day.yield_pct.values[-1] / 100.0))
    return float(f(T_years))

# Spot check: what was the 30d rate on 2010-01-04?
test_date = pd.Timestamp('2010-01-04')
for T in [0.083, 0.25, 1.0, 5.0, 10.0]:
    r = get_rate(test_date, T, treasury)
    print(f'  {test_date.date()} T={T:5.2f}y: r = {r*100:.3f}%')

# Spot check: what was the 1y rate during the 2022 hiking cycle?
test_date2 = pd.Timestamp('2022-12-15')
for T in [0.25, 1.0, 2.0, 10.0]:
    r = get_rate(test_date2, T, treasury)
    print(f'  {test_date2.date()} T={T:5.2f}y: r = {r*100:.3f}%')

  2010-01-04 T= 0.08y: r = 0.050%
  2010-01-04 T= 0.25y: r = 0.080%
  2010-01-04 T= 1.00y: r = 0.450%
  2010-01-04 T= 5.00y: r = 2.650%
  2010-01-04 T=10.00y: r = 3.850%
  2022-12-15 T= 0.25y: r = 4.340%
  2022-12-15 T= 1.00y: r = 4.650%
  2022-12-15 T= 2.00y: r = 4.230%
  2022-12-15 T=10.00y: r = 3.440%


In [12]:
# ================================================================
# 2.2 Persist Treasury curve and verify coverage of OptionsDX dates
# ================================================================
# Save once, read many times during de-Americanisation. Validate every
# OptionsDX trading date has a Treasury observation within 5 days.

treasury_path = AUX_DIR / 'treasury_curve_history.parquet'
treasury.to_parquet(treasury_path)
print(f'Saved: {treasury_path} ({treasury_path.stat().st_size / 1e3:.0f} KB)')

# Build the set of all OptionsDX trading dates across 2010-2023
all_dates = set()
for year in range(2010, 2024):
    df_y = pd.read_parquet(LOCAL_RAW / f'spy_eod_{year}.parquet',
                            columns=['[QUOTE_DATE]'])
    df_y.columns = [c.strip('[]') for c in df_y.columns]
    all_dates.update(pd.to_datetime(df_y['QUOTE_DATE']).dt.date.unique())

# How many SPY dates have a same-day Treasury observation?
treasury_dates = set(treasury.date.dt.date.unique())
matched = all_dates & treasury_dates
unmatched = all_dates - treasury_dates
print(f'\nSPY trading dates: {len(all_dates):,}')
print(f'  Same-day Treasury: {len(matched):,} ({len(matched)/len(all_dates):.1%})')
print(f'  No same-day match: {len(unmatched):,}')
if unmatched:
    print(f'  Sample unmatched: {sorted(unmatched)[:10]}')

Saved: /content/drive/MyDrive/Vol_fitter/data_raw_aux/treasury_curve_history.parquet (454 KB)

SPY trading dates: 3,508
  Same-day Treasury: 3,508 (100.0%)
  No same-day match: 0


In [14]:
# ================================================================
# 6. Wrap-up: promote shared functions to a volsurf package on Drive
# ================================================================
# A few cells of loader code become a .py module that every notebook imports.
# This is the structure the GitHub repo will use, set up now rather than later.

VOLSURF_PKG = PROJECT_ROOT / 'volsurf'
VOLSURF_PKG.mkdir(exist_ok=True)

(VOLSURF_PKG / '__init__.py').write_text('"""Vol surface fitter package."""\n')

(VOLSURF_PKG / 'paths.py').write_text('''"""Canonical paths for the project. Single source of truth."""
from pathlib import Path

PROJECT_ROOT = Path("/content/drive/MyDrive/Vol_fitter")
RAW_DIR = PROJECT_ROOT / "data_SPY"
PROCESSED_DIR = PROJECT_ROOT / "data_processed"
FITTED_DIR = PROJECT_ROOT / "data_fitted"
AUX_DIR = PROJECT_ROOT / "data_raw_aux"
LOCAL_RAW = Path("/content/data_SPY")  # session-scoped staging
''')

(VOLSURF_PKG / 'data.py').write_text('''"""OptionsDX loader."""
from pathlib import Path
import pandas as pd
from .paths import RAW_DIR, LOCAL_RAW

def load_optionsdx_year(year: int, raw_dir: Path = None) -> pd.DataFrame:
    """Load one year of OptionsDX SPY EOD data with normalised schema."""
    raw_dir = raw_dir or (LOCAL_RAW if LOCAL_RAW.exists() else RAW_DIR)
    fp = raw_dir / f"spy_eod_{year}.parquet"
    if not fp.exists():
        raise FileNotFoundError(f"No raw file for year {year}: {fp}")
    df = pd.read_parquet(fp)
    df.columns = [c.strip("[]") for c in df.columns]
    df["QUOTE_DATE"] = pd.to_datetime(df["QUOTE_DATE"])
    df["EXPIRE_DATE"] = pd.to_datetime(df["EXPIRE_DATE"])
    df["T_years"] = df["DTE"] / 365.0
    return df
''')

(VOLSURF_PKG / 'rates.py').write_text('''"""Risk-free rate curve from FRED Treasury data."""
import numpy as np
import pandas as pd
from scipy.interpolate import interp1d
from .paths import AUX_DIR

TREASURY_PATH = AUX_DIR / "treasury_curve_history.parquet"

def load_treasury_curve() -> pd.DataFrame:
    """Load the persisted FRED Treasury curve history."""
    return pd.read_parquet(TREASURY_PATH)

def get_rate(date: pd.Timestamp, T_years: float, treasury_df: pd.DataFrame) -> float:
    """Continuously-compounded risk-free rate via linear interpolation in maturity."""
    day = treasury_df[treasury_df.date == pd.Timestamp(date)]
    if day.empty:
        day = treasury_df[(treasury_df.date <= pd.Timestamp(date)) &
                          (treasury_df.date >= pd.Timestamp(date) - pd.Timedelta(days=5))]
        if day.empty:
            return np.nan
        day = day[day.date == day.date.max()]
    day = day.sort_values("maturity_years")
    if len(day) < 2:
        return np.nan
    f = interp1d(day.maturity_years.values, day.yield_pct.values / 100.0,
                 kind="linear", bounds_error=False,
                 fill_value=(day.yield_pct.values[0] / 100.0,
                             day.yield_pct.values[-1] / 100.0))
    return float(f(T_years))
''')

print("Created volsurf package:")
for f in sorted(VOLSURF_PKG.glob("*.py")):
    print(f"  {f.relative_to(PROJECT_ROOT)}  ({f.stat().st_size} bytes)")

Created volsurf package:
  volsurf/__init__.py  (34 bytes)
  volsurf/data.py  (715 bytes)
  volsurf/paths.py  (376 bytes)
  volsurf/rates.py  (1237 bytes)


In [15]:
# ================================================================
# 7. End-of-notebook manifest: document what's in each location
# ================================================================
# A small JSON snapshot of the data state. Useful for the README, useful
# for spotting drift if someone (you) wonders six months later "what
# was actually in that folder".

import json
from datetime import datetime

manifest = {
    "generated": datetime.now().isoformat(timespec='seconds'),
    "raw_optionsdx": {
        "location": str(RAW_DIR),
        "files": sorted(f.name for f in RAW_DIR.glob("spy_eod_*.parquet")),
        "total_rows": int(stats_df.rows.sum()),
        "trading_days": int(stats_df.trading_days.sum()),
        "year_range": [int(stats_df.year.min()), int(stats_df.year.max())],
        "schema_columns": list(schemas[2010]),
    },
    "treasury": {
        "location": str(treasury_path),
        "rows": len(treasury),
        "date_range": [str(treasury.date.min().date()), str(treasury.date.max().date())],
        "series": sorted(TREASURY_SERIES.keys()),
        "match_with_spy_dates": "100%",
    },
    "outputs_pending": {
        "processed": str(PROCESSED_DIR),
        "fitted": str(FITTED_DIR),
    }
}

manifest_path = AUX_DIR / "data_manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2))
print(f"Manifest written: {manifest_path}\n")
print(json.dumps(manifest, indent=2))

Manifest written: /content/drive/MyDrive/Vol_fitter/data_raw_aux/data_manifest.json

{
  "generated": "2026-06-08T11:00:37",
  "raw_optionsdx": {
    "location": "/content/drive/MyDrive/Vol_fitter/data_SPY",
    "files": [
      "spy_eod_2010.parquet",
      "spy_eod_2011.parquet",
      "spy_eod_2012.parquet",
      "spy_eod_2013.parquet",
      "spy_eod_2014.parquet",
      "spy_eod_2015.parquet",
      "spy_eod_2016.parquet",
      "spy_eod_2017.parquet",
      "spy_eod_2018.parquet",
      "spy_eod_2019.parquet",
      "spy_eod_2020.parquet",
      "spy_eod_2021.parquet",
      "spy_eod_2022.parquet",
      "spy_eod_2023.parquet"
    ],
    "total_rows": 9468584,
    "trading_days": 3508,
    "year_range": [
      2010,
      2023
    ],
    "schema_columns": [
      "QUOTE_UNIXTIME",
      "QUOTE_READTIME",
      "QUOTE_DATE",
      "QUOTE_TIME_HOURS",
      "UNDERLYING_LAST",
      "EXPIRE_DATE",
      "EXPIRE_UNIX",
      "DTE",
      "C_DELTA",
      "C_GAMMA",
      "C_VEGA",
